# Unidad II — AE2: Preparación de datos y Data Mart

**Estudiante:** David Rafael Aguilar Solis · **Matrícula:** 22300607 · **Grupo:** 10IDGS-G1

Este cuaderno documenta y verifica el flujo ya existente. La transformación conservó valores y filas: el producto procesado es una serialización UTF-8/LF validada, no una limpieza que elimine observaciones. No modifica archivos ni ejecuta DDL/ETL.

## 1. Procedencia y protección
La fuente autorizada es el CSV UCI bajo `data/raw`; las salidas derivadas están en `data/processed`. Se leen ambos archivos sin escribirlos, verificando dimensiones, variables, hashes, duplicados y correspondencia exacta de valores.

In [1]:
from pathlib import Path
import hashlib, json, pandas as pd
ROOT=Path.cwd()
RAW=ROOT/'data/raw/ObesityDataSet_raw_and_data_sinthetic.csv'
PROCESSED=ROOT/'data/processed/obesity_unit_ii.csv'
def sha(p): return hashlib.sha256(p.read_bytes()).hexdigest()
raw=pd.read_csv(RAW, encoding='utf-8', on_bad_lines='error')
processed=pd.read_csv(PROCESSED, encoding='utf-8', on_bad_lines='error')
manifest=json.loads((ROOT/'data/processed/unit_ii_manifest.json').read_text(encoding='utf-8'))
assert raw.shape==(2111,17) and processed.shape==(2111,17)
assert raw.columns.tolist()==processed.columns.tolist() and raw.equals(processed)
assert int(raw.duplicated().sum())==24 and int(processed.duplicated().sum())==24
assert not raw.isna().any().any() and not processed.isna().any().any()
print({'raw_sha256':sha(RAW),'processed_sha256':sha(PROCESSED),'rows':len(processed),'columns':len(processed.columns),'exact_duplicates_retained':int(processed.duplicated().sum()),'values_equal':raw.equals(processed),'manifest_keys':list(manifest)})
display(processed.head())

{'raw_sha256': '28b6991062855d3d15642e38b1aa9d1b32fb7abbb5abf5b096f1868cd706154b', 'processed_sha256': '64418dcf666d49d832d07fe53dbfd9415662f2537502fde60e8dbe55b8f5218d', 'rows': 2111, 'columns': 17, 'exact_duplicates_retained': 24, 'values_equal': True, 'manifest_keys': ['executed_at_utc', 'command', 'input_path', 'input_sha256_bytes', 'output_path', 'output_sha256_bytes', 'python_version', 'pandas_version', 'numpy_version', 'rows_read', 'rows_preserved', 'rows_excluded', 'source_values_equal', 'columns', 'column_count', 'missing_values', 'exact_duplicate_rows_preserved', 'transformations', 'decision']}


,Gender,Age,Height,Weight,family_history_with_overweight,FAVC,FCVC,NCP,CAEC,SMOKE,CH2O,SCC,FAF,TUE,CALC,MTRANS,NObeyesdad
0,Female,21.0,1.62,64.0,yes,no,2.0,3.0,Sometimes,no,2.0,no,0.0,1.0,no,Public_Transportation,Normal_Weight
1,Female,21.0,1.52,56.0,yes,no,3.0,3.0,Sometimes,yes,3.0,yes,3.0,0.0,Sometimes,Public_Transportation,Normal_Weight
2,Male,23.0,1.80,77.0,yes,no,2.0,3.0,Sometimes,no,2.0,no,2.0,1.0,Frequently,Public_Transportation,Normal_Weight
3,Male,27.0,1.80,87.0,no,no,3.0,3.0,Sometimes,no,2.0,no,2.0,0.0,Frequently,Walking,Overweight_Level_I
4,Male,22.0,1.78,89.8,no,no,2.0,1.0,Sometimes,no,2.0,no,0.0,0.0,Sometimes,Public_Transportation,Overweight_Level_II


## 2. Decisión de transformación
La validación fila a fila confirma que no se imputaron nulos, no se eliminaron duplicados y no se recodificaron variables. Los 24 duplicados exactos se conservan. La diferencia observada entre archivos es de representación/serialización CSV (raw CRLF frente a salida LF y codificación UTF-8 declarada); no cambia los valores interpretados. Por tanto, el dataset procesado es equivalente y utilizable, pero no debe describirse como si hubiera sufrido una limpieza sustantiva.

In [2]:
print('Columnas:', processed.columns.tolist())
print('Clases:', processed['NObeyesdad'].value_counts().sort_index().to_dict())
print('Nulos:', int(processed.isna().sum().sum()))
print('Bytes raw:', RAW.stat().st_size, 'Bytes procesados:', PROCESSED.stat().st_size)
print('Hash de manifiesto coincide con archivo procesado:', manifest.get('output_sha256_bytes')==sha(PROCESSED))

Columnas: ['Gender', 'Age', 'Height', 'Weight', 'family_history_with_overweight', 'FAVC', 'FCVC', 'NCP', 'CAEC', 'SMOKE', 'CH2O', 'SCC', 'FAF', 'TUE', 'CALC', 'MTRANS', 'NObeyesdad']
Clases: {'Insufficient_Weight': 272, 'Normal_Weight': 287, 'Obesity_Type_I': 351, 'Obesity_Type_II': 297, 'Obesity_Type_III': 324, 'Overweight_Level_I': 290, 'Overweight_Level_II': 290}
Nulos: 0
Bytes raw: 263646 Bytes procesados: 261532
Hash de manifiesto coincide con archivo procesado: True


## 3. Data Mart y estado SQL
El diseño implementado usa staging `DM.StgObesityObservation`, dimensión `DM.DimObesityLevel` y hechos `DM.FactObesityObservation`. La consulta siguiente es exclusivamente de lectura; no crea objetos ni inserta datos. Requiere `sqlcmd` y una instancia local autorizada.

In [3]:
import shutil, subprocess
sqlcmd=shutil.which('sqlcmd')
if sqlcmd:
    query="SET NOCOUNT ON; SELECT 'staging' AS tabla,COUNT_BIG(*) AS filas FROM DM.StgObesityObservation UNION ALL SELECT 'facts',COUNT_BIG(*) FROM DM.FactObesityObservation; SELECT d.ObesityLevel,COUNT_BIG(*) AS filas FROM DM.FactObesityObservation f JOIN DM.DimObesityLevel d ON d.ObesityLevelKey=f.ObesityLevelKey GROUP BY d.ObesityLevel ORDER BY d.ObesityLevel;"
    result=subprocess.run([sqlcmd,'-S',r'.\SQLEXPRESS','-E','-d','REC_ECBD_Obesidad_2026','-W','-s','|','-Q',query],capture_output=True,text=True,timeout=60)
    print('exit_code=',result.returncode); print(result.stdout); print(result.stderr)
    if result.returncode: raise RuntimeError('Consulta SQL de sólo lectura no completada')
else:
    print('NO VERIFICADO: sqlcmd no está disponible en este entorno; el notebook no simula resultados.')

exit_code= 0
tabla|filas
-----|-----
staging|2111
facts|2111
ObesityLevel|filas
------------|-----
Insufficient_Weight|272
Normal_Weight|287
Obesity_Type_I|351
Obesity_Type_II|297
Obesity_Type_III|324
Overweight_Level_I|290
Overweight_Level_II|290




## 4. Carga e idempotencia documentadas
La evidencia previa de Unidad II registra carga real de 2,111 filas en staging/hechos y una segunda carga idempotente. Este cuaderno consulta el estado actual, pero no vuelve a ejecutar ETL ni inserciones para proteger los resultados existentes. Revise `sql/01_create_datamart.sql`, `sql/02_load_datamart.sql` y `docs/unidad_ii/guia_sql_server.md` para definición y procedimiento.

## 5. Limitaciones
El dataset UCI contiene componente sintético reportado por la fuente; las asociaciones son descriptivas y no causales. La igualdad entre raw y processed no sustituye la revisión académica del Data Mart o la aprobación del docente. La entrega/aceptación en Classroom no se infiere de esta ejecución.